In [1]:
import sys
from pathlib import Path
root = Path.cwd().parents[1]
sys.path.append(str(root))

In [2]:
from pprint import pprint
import os
import logging

debug = False
logger = logging.getLogger()
logging.basicConfig(
    level=logging.DEBUG if debug else logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

from src.preprocessing import Reader, FixedCharChunker, Document, Page, Embedder
from src.indexing import DataStore, ChunkStore, FlatVectorStore, Index
from src.utils import AutoEDAIndex

/Users/danilaandreev/Documents/HSE/degree/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Инициализация индекса

In [3]:
DATA_STORE_DIR = '../../data/index/documents'
CHUNK_STORE_DIR = '../../data/index/chunks'
VECTOR_STORE_DIR = '../../data/index/vectors'
SQLITE_DIR = '../../data/index/db'

embedder = Embedder(device='mps')
dimensions = embedder.get_embeddings(['test']).shape[-1]

index = Index(
    datastore=DataStore(DATA_STORE_DIR, logger),
    vectorstore=FlatVectorStore(VECTOR_STORE_DIR, dimensions, logger),
    chunkstore=ChunkStore(CHUNK_STORE_DIR, logger),
    chunker=FixedCharChunker(logger),
    embedder=embedder,
    reader=Reader(logger),
    sqlite_path=SQLITE_DIR,
    logger=logger
)

2026-03-01 15:49:45,601 | INFO | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: sergeyzh/LaBSE-ru-turbo
2026-03-01 15:49:49,166 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sergeyzh/LaBSE-ru-turbo/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-01 15:49:49,252 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sergeyzh/LaBSE-ru-turbo/055975b9638dd075bec75500321f57bc36cb1f4d/modules.json "HTTP/1.1 200 OK"
2026-03-01 15:49:49,447 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sergeyzh/LaBSE-ru-turbo/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-03-01 15:49:49,657 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sergeyzh/LaBSE-ru-turbo/resolve/main/config_sentence_transformers.json "HTTP/1.1 404 Not Found"
2026-03-01 15:49:49,860 | INFO | httpx | HTTP Request: HEAD https://huggingface.co/sergeyzh/LaBSE-ru-turbo/resolve/main/README.md "HT

In [4]:
# index.clear()
index.info()

2026-03-01 15:49:55,202 | INFO | root | Index stats: {'datastore': {'documents': 175}, 'chunkstore': {'chunks': 35525}, 'vectorstore': {'vectors': 35636, 'dimensions': 768}, 'index': {'points': 35525, 'points_without_chunk': 0, 'chunks_without_point': 0}}


{'datastore': {'documents': 175},
 'chunkstore': {'chunks': 35525},
 'vectorstore': {'vectors': 35636, 'dimensions': 768},
 'index': {'points': 35525,
  'points_without_chunk': 0,
  'chunks_without_point': 0}}

## Загрузка документов

In [6]:
SOURCE_DATA_DIR = '../../data/raw_data'

sources = [os.path.join(SOURCE_DATA_DIR, name) for name in os.listdir(SOURCE_DATA_DIR)]
print(f'Количество источников: {len(sources)}')

doc_ids = index.add_sources(sources)

2026-03-01 14:06:10,454 | INFO | root | add_sources: 175 new sources, 0 skipped


Количество источников: 175


Indexing 4293780511.pdf | RSS 1675 MB | MPS 490/4436 MB:  61%|██████    | 106/175 [06:04<02:05,  1.82s/it]2026-03-01 14:12:14,569 | INFO | root | File 8735c923b0c51ca2ccf9bd3bdf0552aaa762a0e627f6daca43cbe6a0f79e44da.json already exists
Indexing 4294850881.pdf | RSS 2073 MB | MPS 490/3467 MB:  90%|█████████ | 158/175 [09:14<01:02,  3.66s/it]2026-03-01 14:15:25,209 | INFO | root | File 3a87da2057da5f074af2abfb0b0833578bf341e10b7696543bcb511439e05e5d.json already exists
2026-03-01 14:15:25,210 | INFO | root | File 02d7943892e1a20f6f425e0ea644e41bd583f8f60d4085543c6a75dc092e0206.json already exists
2026-03-01 14:15:25,211 | INFO | root | File 836db29539c2016f3c853a9fb90d3ad67d9601a9ef5a69b0b30c1af10dbe200f.json already exists
2026-03-01 14:15:25,211 | INFO | root | File cdd00d2b45596f04446f9fcebc743ee96f9af94ec6b027208e53807c8257239c.json already exists
2026-03-01 14:15:25,211 | INFO | root | File 2f805f0e5e16b27004772dab774769c53f7f3cadb32eaddebe7365625d203346.json already exists
2026-03-

In [7]:
index.save_vectorstore()

2026-03-01 14:47:36,080 | INFO | root | Index vectorstore saved


In [8]:
index.info()

2026-03-01 14:47:39,479 | INFO | root | Index stats: {'datastore': {'documents': 175}, 'chunkstore': {'chunks': 35525}, 'vectorstore': {'vectors': 35636, 'dimensions': 768}, 'index': {'points': 35525, 'points_without_chunk': 0, 'chunks_without_point': 0}}


{'datastore': {'documents': 175},
 'chunkstore': {'chunks': 35525},
 'vectorstore': {'vectors': 35636, 'dimensions': 768},
 'index': {'points': 35525,
  'points_without_chunk': 0,
  'chunks_without_point': 0}}

In [5]:
test_query = 'Нагрузка перекрытий'
res = index.search(test_query)

for r in res:
    print(r.chunk['text'])
    print()

едует рассчитывать на нагрузку от перекрытий и на давление от свежеуложенной, неотвердевшей кладки, эквивалентное 53

ельных перекрытий: сплошное загружение принятой нагрузкой; неблагоприятное частичное загружение при расчете конструкций и оснований, чувствительных к такой схеме загружения; отсутствие временной нагрузки. 8.1 Определение нагрузок от оборудования, складируемых материалов и изделий 8.1.1 Нагрузки от оборудования (в том числе трубопроводов, транспортных средств), складируе­ мых материалов и изделий устанавливаются в задании на проектирование на основании технологиче­ ских решений, в которых должны быть приведены: а) возможные на каждом перекрытии и полах на грунте места расположения и габариты опор оборудования, размеры участков складирования и хранения материалов и изделий, места возможного перемещения оборудования в процессе эксплуатации или перепланировки; 5

ания этих конструкций. При этом следует учесть также случай пониженных значений кратковременных нагрузок. 8 Нагр

# EDA

In [5]:
eda = AutoEDAIndex(index)
df = eda.run()
df

DOCUMENTS
- count: 175
- text_length_mean: 140687.2857142857
- text_length_median: 108903.0
- text_length_min: 1874
- text_length_max: 693714

CHUNKS
- count: 35525
- text_length_mean: 824.0530893736805
- text_length_median: 1000.0
- text_length_min: 51
- text_length_max: 1000
- text_length_std: 284.07516424503365
- words_mean: 113.84551724137931
- sentences_mean: 9.412751583391977
- paragraphs_mean: 1.0

EMBEDDINGS
- matrix_rows: 35636
- matrix_cols: 768
- dtype: float64
- norm_mean: 1.0000000280155166
- norm_std: 3.7514336223883655e-08
- norm_min: 0.9999998464061922
- norm_max: 1.0000001507952543
- dimension_std_mean: 0.019282826883777405
- dimension_std_min: 0.010518082034465968
- dimension_std_max: 0.030330688909928495
- pairwise_similarity_pairs: 634944430
- pairwise_similarity_min: 0.31135926838705935
- pairwise_similarity_max: 1.0000001870493243
- pairwise_similarity_mean: 0.7102730506460225
- pairwise_similarity_std: 0.05982386750951728



,section,metric,value
0,documents,count,175
1,documents,text_length_mean,140687.285714
2,documents,text_length_median,108903.0
3,documents,text_length_min,1874
4,documents,text_length_max,693714
5,chunks,count,35525
6,chunks,text_length_mean,824.053089
7,chunks,text_length_median,1000.0
8,chunks,text_length_min,51
9,chunks,text_length_max,1000


После создания бенчмарка требуется сделать таблицу, которая будет показывать все основные метрики качества при различных комбинациях создания индекса. 
- Сначала сделаем бейзлайн
- Потом попробуем перебрать разные варианты чанкирования - выберем лучший для по метрикам
- Потом переберем разные варианты моделей - выберем лучшие
- Потом будет пробовать изменить архитектуру - выбираем лучшую
- Потом допиливаем через дообучение моделей
Всё должно собираться в dataframe для сохранения результатов

In [10]:
index.close()

2026-03-01 14:48:34,623 | INFO | root | Index vectorstore saved
